# Prepare data for the MRT demand map
Turns the raw LTA **Passenger Volume by Train Stations** CSV of each month into the small `.js` files the map reads.

**To add a new month**
1. Download the CSV from LTA DataMall and put it in `prepare/raw/`.
2. Add one entry to `months.json` (month, csv file name, the date you downloaded it). Leave `days` as `null` unless the day counts below look wrong.
3. Make sure the year's public holidays are in `holidays.json`.
4. Run all cells (Kernel > Restart & Run All). Read the warnings in the output of step 2.
5. Reload `index.html`. The new month appears in the Month list.

## Step 1 - what is configured

In [ ]:
import json, pandas as pd
from pathlib import Path
import prepare_data as pdx
print(pdx.HERE)
months = json.load(open('months.json'))['months']
pd.DataFrame(months)

In [ ]:
hol = json.load(open('holidays.json'))
for year, lst in hol.items():
    if year.startswith('_'): continue
    print(year, len(lst), 'holidays listed')
    for h in lst: print('   ', h['date'], h['name'])

## Step 2 - build the data files
Watch the output for lines starting with **WARNING**:
* fewer than 11 holidays listed for the year (day counts may be wrong)
* codes in the CSV that are not in the coordinates workbook, or stations with no coordinates
* total volume more than 25% different from the previous month

In [ ]:
pdx.run()

## Step 3 - check the day counts and totals

In [ ]:
rows = []
for m in json.load(open(pdx.DATA.parent / 'prepare' / 'months.json'))['months']:
    mo = m['month']
    raw = pd.read_csv(pdx.HERE / 'raw' / m['csv'], encoding='utf-8-sig')
    for dtype in ['WEEKDAY', 'WEEKENDS/HOLIDAY']:
        sub = raw[raw.DAY_TYPE == dtype]
        rows.append({'month': mo, 'day type': dtype, 'raw tap-ins': int(sub.TOTAL_TAP_IN_VOLUME.sum()), 'raw tap-outs': int(sub.TOTAL_TAP_OUT_VOLUME.sum())})
pd.DataFrame(rows)

In [ ]:
# manifest as the page will see it
print(Path(pdx.DATA / 'manifest.js').read_text())

## Notes
* Stations and their coordinates come from `MRT_Stations_with_Coordinates.xlsx`, joined on `PT_CODE`. The LATITUDE/LONGITUDE columns inside the LTA CSV are ignored.
* Hour 0 in the raw file is dropped; the map shows hours 5 to 23.
* Every figure is the monthly total divided by the number of days of that type. Weekend / holiday = Saturdays, Sundays and public holidays on weekdays.
* To change which line a station belongs to, edit `PREFIX_TO_CHIP` in `prepare_data.py`.